# Turkish Part-of-Speech Tagging with a Hidden Markov Model

**Overview.** A first-order hidden Markov model (HMM) part-of-speech tagger for Turkish, implemented from scratch: transition and emission probabilities are estimated with maximum likelihood and tag sequences are decoded with the Viterbi algorithm. The tagger is trained and evaluated on two tag sets, the full Universal POS (UPOS) inventory and a reduced set of five tags.

**Dataset.** [Google Research Turkish Treebanks](https://github.com/google-research-datasets/turkish-treebanks) (`web.conllu` + `wiki.conllu`, CoNLL-U format, Apache 2.0). The two files are combined into 4,851 sentences and split 80/20 into 3,880 training and 971 test sentences.

**Approach.**
- Each token is represented by its lemma (CoNLL-U column 3) and its UPOS tag; sub-token segments without a lemma (`_`) are skipped.
- MLE counts for tag-to-tag transitions (including `<START>` and `<END>`), tag-to-word emissions and tag frequencies.
- Viterbi decoding in log space. Unseen transitions get a log-probability floor of -1e9 and unseen word/tag pairs a constant emission probability of 1e-6, so context decides the tag of unknown words.
- Two settings: **A**, all 14 UPOS tags; **B**, `ADJ`, `ADV`, `NOUN`, `VERB`, `PUNCT` kept and every other tag mapped to `OTHER`.
- Token-level accuracy, weighted F1 and confusion matrices on the held-out test set.

**Key results.**

| Tag set | Accuracy | Weighted F1 |
|---|---|---|
| A: all tags | 0.9068 | 0.9063 |
| B: reduced (5 tags + OTHER) | 0.9162 | 0.9156 |

The largest remaining errors in both settings are confusions between content-word classes: NOUN vs. VERB and NOUN vs. ADJ.

## 1. Data loading

The two CoNLL-U files are parsed sentence by sentence. Comment lines are skipped, the lemma (column 3) is used as the observed word and the UPOS tag (column 4) as the hidden state. Rows whose lemma is `_` (derivational sub-token segments) are dropped. The web and Wikipedia portions are concatenated into one corpus.

In [2]:
import os

def read_conll(file_path):
    sentences = []
    sentence = []

    # Check if file exists before opening
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # Skip metadata lines starting with "#"
            if line.startswith("#"):
                continue

            # Sentence boundary 
            if not line:
                if sentence:
                    sentences.append(sentence)
                    sentence = []
            else:
                # Split the line into columns and extract word and POS tag
                columns = line.split('\t')
                if len(columns) > 3:  # Check to ensure the line has expected columns
                    word = columns[2] # LEMMA column (FORM is at index 1)
                
                    if word == '_':
                      continue

                    pos_tag = columns[3] # UPOS
                    sentence.append((word, pos_tag))

        # Add the last sentence if the file doesn't end with a blank line
        if sentence:
            sentences.append(sentence)

    return sentences


web_file_path = "web.conllu" 
wiki_file_path = "wiki.conllu"


web_sentences = read_conll(web_file_path)
wiki_sentences = read_conll(wiki_file_path)

all_sentences = web_sentences + wiki_sentences
print(f"Total sentences loaded: {len(all_sentences)}")

Total sentences loaded: 4851


## 2. Tag-set preparation

`prepare_data` returns the corpus unchanged for the full tag set (setting A). For the reduced tag set (setting B), the tags `ADJ`, `ADV`, `NOUN`, `VERB` and `PUNCT` are kept and every other tag is mapped to `OTHER`, so sentence length and structure are preserved.

In [3]:
def prepare_data(sentences, tag_list=None):
    
    if tag_list is None:
        # Scenario A: Use all tags, no filtering needed
        return sentences

    prepared_sentences = []
    
    # Convert list to set for faster lookup
    allowed_tags = set(tag_list)

    for sentence in sentences:
        new_sentence = []
        for word, tag in sentence:
            # If the tag is in our allowed list, keep it.
            # Otherwise, map it to 'OTHER' so we maintain sentence structure 
            # but don't distinguish between the excluded tags.
            if tag in allowed_tags:
                new_sentence.append((word, tag))
            else:
                new_sentence.append((word, 'OTHER'))
        
        prepared_sentences.append(new_sentence)

    return prepared_sentences


## 3. Train / test split (80/20, `random_state=42`)

The split is done at sentence level on the raw corpus, before tag-set preparation, so both settings are trained and evaluated on exactly the same sentences.

In [4]:
from sklearn.model_selection import train_test_split

# split the raw data first and will apply prepare_data later 
# depending on whether running Scenario A or B.
train_data, test_data = train_test_split(all_sentences, test_size=0.2, random_state=42)

print(f"Training set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")

Training set size: 3880
Test set size: 971


## 4. HMM training (MLE counts)

`create_HMM` collects the sufficient statistics for maximum-likelihood estimation:

- **Transitions:** `count(prev_tag -> tag)`, with a `<START>` tag before the first word and an `<END>` tag after the last word of every sentence.
- **Emissions:** `count(tag, word)`.
- **Tag counts:** the denominators for both distributions, including the number of sentences as the count of `<START>`.

The probabilities are `P(tag | prev_tag) = count(prev_tag -> tag) / count(prev_tag)` and `P(word | tag) = count(tag, word) / count(tag)`. They are computed from these counts during decoding.

In [5]:
from collections import defaultdict

def create_HMM(train_sentences):
    # Initialize dictionaries for counts
    # Using defaultdict makes it easier to increment counts without checking if key exists
    transitions = defaultdict(int) 
    emissions = defaultdict(int)
    tag_counts = defaultdict(int)
    vocab = set()
    
    # We need a special Start tag to calculate probability of a tag starting a sentence
    START_TAG = "<START>"
    END_TAG = "<END>"
    
    for sentence in train_sentences:
        prev_tag = START_TAG
        
        for word, tag in sentence:
            # Update Vocabulary
            vocab.add(word)
            
            # Count Emission: P(word | tag)
            emissions[(tag, word)] += 1
            
            # Count Transition: P(tag | prev_tag)
            transitions[(prev_tag, tag)] += 1
            
            # Count Tag occurrences 
            tag_counts[tag] += 1
            
            # Update previous tag for next iteration
            prev_tag = tag
            
        # Handle the End of sentence transition
        transitions[(prev_tag, END_TAG)] += 1
        
        # Also need to count occurrences of the START tag to normalize start transitions
        tag_counts[START_TAG] += 1
        # No need to strictly count END_TAG as a denominator for transitions, 
        # because nothing transitions FROM END

    word_count = len(vocab)
    
    # Convert defaultdicts to standard dicts for clean return
    return dict(transitions), dict(emissions), dict(tag_counts), vocab, word_count

## 5. Viterbi decoding

The most likely tag sequence for each test sentence is found with the Viterbi algorithm in log space, to avoid numerical underflow:

1. **Initialisation:** `log P(tag | <START>) + log P(word_1 | tag)`.
2. **Recursion:** for every position and tag, keep the best-scoring previous tag and a backpointer to it.
3. **Termination:** add `log P(<END> | tag)` and pick the best final tag.
4. **Backtracking:** follow the backpointers to recover the full sequence.

Unseen transitions receive a log-probability floor of -1e9. Unseen word/tag pairs (unknown words) receive a constant emission probability of 1e-6, which leaves the decision to the transition model.

In [6]:
import numpy as np
import math

def viterbi(test_sentence, transitions, emissions, tag_counts, word_count):
   
    # Get unique tags
    # Filter out START/END from the potential output tags for the words
    unique_tags = [t for t in tag_counts.keys() if t not in ["<START>", "<END>"]]
    num_tags = len(unique_tags)
    len_sent = len(test_sentence)
    
    # Small constant for smoothing log(0)
    # log(0) is -inf, so we use a very small number like -1e9
    SMOOTHING_LOG = -1e9
    
    # Initialize Viterbi matrix (Rows: Tags, Cols: Words in sentence)
    v_matrix = np.full((num_tags, len_sent), -np.inf)
    
    # Backpointer matrix to reconstruct path
    # Stores the index of the best previous tag
    backpointer = np.zeros((num_tags, len_sent), dtype=int)
    
    #  1. Initialization Step
    first_word = test_sentence[0]
    
    for i, tag in enumerate(unique_tags):
        # Transition: P(tag | <START>)
        # MLE estimate (simple ratio of counts)
        # P(tag|start) = count(start->tag) / count(start)
        trans_count = transitions.get(("<START>", tag), 0)
        start_total = tag_counts.get("<START>", 1)
        
        # Emission: P(word | tag)
        emit_count = emissions.get((tag, first_word), 0)
        tag_total = tag_counts.get(tag, 1)
        
        # Handling Log Probs
        if trans_count > 0:
            trans_prob = math.log(trans_count / start_total)
        else:
            trans_prob = SMOOTHING_LOG 

        # Emission Probability with Unknown Word Handling
        # If word is unseen (count 0), don't punish it with -inf.
        # Give it a small probability so context decides.
        if emit_count > 0:
            emit_prob = math.log(emit_count / tag_total)
        else:
            # Strategy for unknown words: 
            # If the word is unknown, assign a uniform prob or small constant
            emit_prob = math.log(1e-6) 
            
        v_matrix[i, 0] = trans_prob + emit_prob

    #  2. Recursion Step
    for t in range(1, len_sent):
        word = test_sentence[t]
        
        for curr_idx, curr_tag in enumerate(unique_tags):
            best_prob = -np.inf
            best_prev_idx = -1
            
            # Calculate Emission P(word | curr_tag) once per column
            emit_count = emissions.get((curr_tag, word), 0)
            tag_total = tag_counts.get(curr_tag, 1)
            
            if emit_count > 0:
                emit_prob = math.log(emit_count / tag_total)
            else:
                emit_prob = math.log(1e-6) # Unknown word handling

            # Check transition from every possible previous tag
            for prev_idx, prev_tag in enumerate(unique_tags):
                
                # Transition: P(curr_tag | prev_tag)
                trans_count = transitions.get((prev_tag, curr_tag), 0)
                prev_tag_total = tag_counts.get(prev_tag, 1)
                
                if trans_count > 0:
                    trans_prob = math.log(trans_count / prev_tag_total)
                else:
                    trans_prob = SMOOTHING_LOG
                
                # Calculate probability: Prev_Path + Transition + Emission
                current_score = v_matrix[prev_idx, t-1] + trans_prob + emit_prob
                
                if current_score > best_prob:
                    best_prob = current_score
                    best_prev_idx = prev_idx
            
            v_matrix[curr_idx, t] = best_prob
            backpointer[curr_idx, t] = best_prev_idx

    # 3. Termination Step 
    # Must account for transition to <END>
    best_final_score = -np.inf
    best_final_tag_idx = -1
    
    for i, tag in enumerate(unique_tags):
        # P(<END> | tag)
        trans_count = transitions.get((tag, "<END>"), 0)
        tag_total = tag_counts.get(tag, 1)
        
        if trans_count > 0:
            end_prob = math.log(trans_count / tag_total)
        else:
            end_prob = SMOOTHING_LOG
            
        score = v_matrix[i, len_sent-1] + end_prob
        
        if score > best_final_score:
            best_final_score = score
            best_final_tag_idx = i

    #  4. Backtracking 
    # Follow the backpointers to recover the path
    predicted_indices = [best_final_tag_idx]
    
    for t in range(len_sent - 1, 0, -1):
        # Get the previous tag index stored in backpointer
        prev_idx = backpointer[predicted_indices[-1], t]
        predicted_indices.append(prev_idx)
        
    predicted_indices.reverse()
    
    # Map indices back to tag strings
    predicted_tags = [unique_tags[idx] for idx in predicted_indices]
    
    return predicted_tags

## 6. Evaluation

Both settings are trained on the training split and evaluated on the test split with token-level accuracy, weighted F1 and a confusion matrix of gold vs. predicted tags.

In [9]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def run_evaluation(scenario_name, target_tags=None):
    print(f"\n{'='*20} Running Scenario: {scenario_name} {'='*20}")
    
    # 1. Prepare Data
    print("Data preparation")
    train_prep = prepare_data(train_data, target_tags)
    test_prep = prepare_data(test_data, target_tags)
    
    # 2. Train Model
    print("Training HMM")
    transitions, emissions, tag_counts, vocab, word_count = create_HMM(train_prep)
    
    # 3. Prediction
    print("Predicting on test set")
    all_gold_tags = []
    all_pred_tags = []
    
    for sentence in test_prep:
        # Extract just words for prediction
        words = [pair[0] for pair in sentence]
        gold_tags = [pair[1] for pair in sentence]
        
        try:
            preds = viterbi(words, transitions, emissions, tag_counts, word_count)
            
            # Aggregate results
            all_gold_tags.extend(gold_tags)
            all_pred_tags.extend(preds)
        except Exception as e:
            print(f"Error in sentence: {words}")
            print(e)
            break

    # 4. Metrics
    acc = accuracy_score(all_gold_tags, all_pred_tags)
    f1 = f1_score(all_gold_tags, all_pred_tags, average='weighted')
    
    print(f"\nResults for {scenario_name}:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score (Weighted): {f1:.4f}")
    
    # 5. Confusion Matrix
    # Get unique labels sorted
    labels = sorted(list(set(all_gold_tags + all_pred_tags)))
    cm = confusion_matrix(all_gold_tags, all_pred_tags, labels=labels)
    
    print("\nConfusion Matrix:")
    # Using a DataFrame for better readability in output
    df_cm = pd.DataFrame(cm, index=labels, columns=labels)
    print(df_cm)
    
    return acc, f1

# --- Run Scenario A: All Tags ---
acc_a, f1_a = run_evaluation("Scenario A (All Tags)", target_tags=None)

# --- Run Scenario B: Restricted Tags ---
# "ADJ, ADV, NOUN, VERB, PUNCT"
target_list = ["ADJ", "ADV", "NOUN", "VERB", "PUNCT"]
acc_b, f1_b = run_evaluation("Scenario B (Restricted Tags)", target_tags=target_list)

# --- Comparison ---

print("FINAL COMPARISON")

print(f"Scenario A (All Tags)      -> Accuracy: {acc_a:.4f}, F1: {f1_a:.4f}")
print(f"Scenario B (Restricted)    -> Accuracy: {acc_b:.4f}, F1: {f1_b:.4f}")



==================== Running Scenario: Scenario A (All Tags) ====================
Data preparation
Training HMM
Predicting on test set

Results for Scenario A (All Tags):
Accuracy: 0.9068
F1 Score (Weighted): 0.9063

Confusion Matrix:
       ADJ  ADP  ADV  CONJ  DET  NOUN  NUM  ONOM  PRON  PRT  PUNCT  VERB   X
ADJ    407    7   21     1    1   118    0     0     5    0      3    11   0
ADP      3  472    3     0    2    48    0     0     0    0      0     3   0
ADV     25   10  249     3    7    21    0     0     2    0      0     8   0
CONJ     0   11    2   517    0     6    0     0     6   10      1     0   6
DET      1    0   10     1  437     0    0     0    16    0      0     6   1
NOUN    76   73   13     1    7  5349   32     0     3    1     45   202   3
NUM      0    0    0     0   12    55  214     0     0    0      0     6   0
ONOM     0    0    0     0    0     1    0     0     0    0      0     0   0
PRON     9    0    1     1   51     4    0     0   190    0      0     

## 7. Discussion & Results

### Data preparation and MLE estimation
The `web.conllu` and `wiki.conllu` files were merged and split into training (80%) and test (20%) sets with a fixed `random_state` for reproducibility. For the reduced tag-set setting, a filtering function keeps `ADJ`, `ADV`, `NOUN`, `VERB` and `PUNCT` intact and maps all other tags (such as `DET`, `PRON` and `CONJ`) to a single `OTHER` label.

Probabilities are estimated with maximum likelihood. Tag-to-tag transitions and tag-to-word emissions are counted in dictionaries. The transition probability is the count of a specific transition divided by the total count of the previous tag, and the emission probability is the count of a word appearing with a tag divided by the total count of that tag. All computations are done with log probabilities to prevent the numerical underflow that comes from multiplying many small probabilities.

### Challenges
The main implementation challenge was handling events in the Viterbi algorithm that never occur in the training data. Taking the logarithm of zero for unseen transitions or words causes math errors. This is resolved by assigning a very small log probability to these events instead of zero.

On the data side, the main challenge is the ambiguity of content words. The confusion matrices show that the model frequently struggles to distinguish nouns from adjectives. This is likely due to the nature of Turkish, where adjectives often function as nouns, which makes the boundary hard for a simple statistical model to detect without deeper semantic context.

### Performance
Simplifying the tag set helps the HMM: the reduced-tag version outperforms the all-tags version by about one percentage point in accuracy (91.62% vs. 90.68%). The main reason is that similar tags such as determiners and pronouns are grouped into a single `OTHER` category, which removes the confusion between them. Even with this grouping, the core difficulty remains: both models still struggle to separate nouns, verbs and adjectives. The real difficulty therefore lies not in the small function words but in resolving the ambiguity of the main content words.

### Limitations and possible improvements
The main limitation is the first-order Markov assumption, which uses only the immediately preceding tag to predict the next one. This ignores longer-range dependencies in a sentence that could help disambiguate complex phrases. The handling of unknown words is also relatively simple.

A significant improvement would be to incorporate morphological analysis. Turkish is agglutinative, so the suffix of an unknown word usually gives a strong clue about its part of speech. A suffix-based feature for unknown words would likely reduce the confusion between nouns and verbs significantly.